# IOAI — 2025 Contest Underfitting Cv (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, glob, zipfile
os.system('pip -q install timm')
if not os.path.exists('data/train.csv'):
    os.makedirs('data', exist_ok=True)
    os.environ['KAGGLE_API_TOKEN'] = 'KGAT_5dd261fded8e0d7eb2c29d8d65dfabea'  # 내장 토큰(규칙 수락된 계정)
    os.system('pip -q install kaggle')
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    api.competition_download_files('neoai-2025-underfitting-cv', path='data', quiet=False)
    for zp in glob.glob('data/*.zip'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data')
        os.remove(zp)
print('데이터 준비:', 'data/train.csv' if os.path.exists('data/train.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# NeoAI 2025 — Underfitting CV

> **Northern Eurasia Olympiad in AI 2025 · Kaggle playground competition (이미지 분류 · 전이학습)**

**핵심 상황**: 테스트 5,100장은 **~102개 클래스**인데, 라벨이 있는 **학습 이미지는 230장(23개 클래스)뿐**입니다.
그래서 밑바닥부터 학습하면 *과소적합(underfitting)* + 못 본 클래스 처리 불가 → 점수가 무너집니다. 대회는
**일부러 덜 학습시킨(underfit) 모델 `model.pt`**(timm `tiny_vit_5m_224`, 102클래스 head)를 제공합니다.
지표 = **정확도(accuracy)**.

**채점**: test 라벨은 비공개(102클래스)라 로컬 held-out(23클래스)으로는 제대로 못 잽니다. → 사이트 **Submissions**
탭에서 `submission.csv`(`id,class`)를 **캐글 리더보드에 자동 제출**해 실제 정확도를 받아옵니다(일일 제출 한도 15회).

⚠️ **아래 베이스라인 = 제공된 underfit 모델을 그대로 추론**(정확도 ≈ 0.33). 미세조정으로 underfitting 을
완화하면 크게 오릅니다(모범답안 ≈ 0.67). **전처리 주의**: 이 모델은 ImageNet 정규화가 아니라 **0–1 스케일(정규화
없음)**로 학습됨 → `ToTensor()` 만 사용(정규화 넣으면 정확도 급락).

In [ ]:
import torch, timm, pandas as pd, numpy as np, os
from PIL import Image
import torchvision.transforms as T
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device', dev)

def load_model():
    m = timm.create_model('tiny_vit_5m_224', num_classes=102)
    sd = torch.load('data/model.pt', map_location='cpu', weights_only=False)
    m.load_state_dict({k[len('model.'):]: v for k, v in sd.items()})   # 'model.' 접두 제거
    return m.to(dev)

# 정규화 없이 224×224 (이 모델의 학습 전처리)
tf_eval = T.Compose([T.Resize((224, 224)), T.ToTensor()])
test_files = sorted(os.listdir('data/test_images'))
print('test images:', len(test_files))

def predict(model, files, folder, tf, bs=128):
    model.eval(); out = []
    with torch.no_grad():
        for i in range(0, len(files), bs):
            xb = torch.stack([tf(Image.open(f'{folder}/{f}').convert('RGB')) for f in files[i:i+bs]]).to(dev)
            out.append(model(xb).argmax(1).cpu().numpy())
    return np.concatenate(out)


In [ ]:
# BASELINE: underfit 모델을 그대로 추론 (미세조정 없음)
m = load_model()
pred = predict(m, test_files, 'data/test_images', tf_eval)
pd.DataFrame({'id': test_files, 'class': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(test_files), '| 예측 클래스 수', len(set(pred)),
      '\n→ Submissions 탭에서 캐글 채점 (베이스라인 ≈ 0.33). 미세조정으로 올리세요(모범답안 참고)')


## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)